# Task A Run 40 -- does a second model family improve the Gemma ensemble?

Every gain since Gemma has come from **averaging**, and averaging gains most when members
make *different* mistakes. Seeds, TAPT runs and R-Drop all vary the same Gemma-4-12B, so its
members share its blind spots. A different architecture, with its own tokenizer and
pretraining data, is the standard source of more diverse errors.

**Everything is trained in this notebook; nothing to attach, nothing to edit.** All models
train on the 6,114-row split and are scored on the 1,079-row holdout (`815110ff24`):

| arm | what |
|---|---|
| **reference: Gemma TAPT x2** | the final recipe: TAPT (1 epoch, training-split text + external corpus), then two 3-epoch Gemma-4-12B classifiers (seeds 42, 43) from that adapter |
| **Qwen x2** | Qwen3-8B-Base with Gemma's classifier recipe (4-bit QLoRA, 3 epochs, lr 1e-4, LoRA r16, short prompt), seeds 42, 43. Second-best reader of Kanglish in Run 20's screen (3.19 bits per character against Gemma's 3.01) |
| **candidate: Gemma TAPT + Qwen** | all four models averaged with **equal weight per model**, as `combine_probs` builds the final. No weight is tuned on the holdout |

**Verdict.** A paired bootstrap of the candidate against the reference.

| result | meaning |
|---|---|
| CI lower bound above 0 | **significant**: build a full-data Qwen fit and a Task B version |
| P(better) >= 0.7 | a probable gain, but inside the noise; optional |
| otherwise | stop and keep Gemma only |

**Time:** about 6 h. GPU 0 runs TAPT and then one Gemma classifier; GPU 1 runs both Qwen
fits and then the other Gemma classifier. The Qwen holdout models also predict the test set.

**Settings:** GPU T4 x2, Internet on. No token and no attached input are needed.
Save & Run All. A hard stop at 11 h means the notebook always finishes and packages
what it has.

In [ ]:
import json, os, pathlib, subprocess, sys, time
from fnmatch import fnmatch
T_START = time.time()
HARD_STOP = T_START + 11 * 3600   # Kaggle kills at 12 h and then saves nothing

REPO = "https://github.com/robinpnalex/Hastika-ICON2026.git"
BRANCH = "task-b"
WORK = "/tmp/hastika"             # outside /kaggle/working: the output holds results only


def git(*args):
    return subprocess.run(["git", "-C", WORK, *args], check=True,
                          capture_output=True, text=True).stdout.strip()


try:
    remote = subprocess.run(["git", "ls-remote", "--exit-code", REPO, f"refs/heads/{BRANCH}"],
                            check=True, capture_output=True, text=True,
                            timeout=60).stdout.split()[0]
except Exception as e:
    raise SystemExit("Cannot reach GitHub. Turn on Settings -> Internet, then restart.") from e
if os.path.isdir(WORK + "/.git"):
    git("fetch", "-q", "--depth", "1", "origin", BRANCH)
    git("reset", "-q", "--hard", "FETCH_HEAD")
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, "--depth", "1", REPO, WORK], check=True)
head = git("rev-parse", "HEAD")
assert head == remote, f"clone is at {head[:8]} but {BRANCH} is at {remote[:8]}"
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
print("repo at", git("log", "-1", "--oneline"))
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass                              # Gemma-4 is not gated; no token is needed

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score

assert torch.cuda.is_available(), "select GPU T4 x2"
N_GPU = torch.cuda.device_count()
print("gpus:", [torch.cuda.get_device_name(i) for i in range(N_GPU)])
OUT = pathlib.Path("/kaggle/working/a40_outputs")
RUNS, LOGS, SUB = OUT / "runs", OUT / "logs", OUT / "submissions"
for p in (RUNS, LOGS, SUB):
    p.mkdir(parents=True, exist_ok=True)
HF_CACHE = "/tmp/hf"
os.environ["HF_HUB_CACHE"] = HF_CACHE

from hastika.common.gpu_queue import run_queue
from hastika.task_b.combined_holdout import fingerprint, paired_bootstrap

TASK = "a"
CLASSES = ["Non-Hate", "Hate"]
LABEL_COL = "Label"
SPLIT = pathlib.Path(WORK) / "data/derived/task_a_combined_holdout"
df = pd.read_csv(SPLIT / "all.csv")
holdout = pd.read_csv(SPLIT / "holdout.csv")
assert (len(df), len(holdout)) == (7193, 1079), (len(df), len(holdout))
HOLDOUT_FP = fingerprint(holdout["id"])
assert HOLDOUT_FP == "815110ff24", HOLDOUT_FP
y_ho = holdout[LABEL_COL].map(CLASSES.index).to_numpy()
TEST_CSV = pathlib.Path("/tmp/test_inputs.csv")
pd.read_csv(pathlib.Path(WORK) / "data/raw/hastika_binary_test.csv")[["id", "Comment"]].to_csv(TEST_CSV, index=False)
test = pd.read_csv(TEST_CSV)
EXTERNAL = pathlib.Path(WORK) / "data/external/offenseval_kn.csv"
print(f"task {TASK.upper()}: train {len(df) - len(holdout)}, holdout {len(holdout)} ({HOLDOUT_FP}), "
      f"all {len(df)}, test {len(test)}")


def report(y, probs, name):
    pred = probs.argmax(1)
    per = f1_score(y, pred, average=None, labels=range(len(CLASSES)), zero_division=0)
    macro = float(f1_score(y, pred, average="macro"))
    print(f"{name:34s} macro-F1 {macro:.4f}  acc {accuracy_score(y, pred):.4f}  | "
          + "  ".join(f"{c[:5]} {f:.3f}" for c, f in zip(CLASSES, per)))
    return macro


def boot_line(label, a, b):
    r = paired_bootstrap(y_ho, a, b)
    print(f"{label:28s} {r['diff']:+.4f}  95% CI [{r['ci95'][0]:+.4f}, {r['ci95'][1]:+.4f}]  "
          f"P(better) {r['p_better']:.2f}")
    return r


# Earlier runs' outputs are reused when attached (Add Input -> Notebook Output), found by
# walking every input with symlinks followed. Nothing below *requires* them.
def find_prior(pattern):
    hits = []
    for root, _, files in os.walk("/kaggle/input", followlinks=True):
        for f in files:
            p = os.path.join(root, f)
            if fnmatch(p, pattern):
                hits.append(pathlib.Path(p))
    return sorted(hits)


def show_inputs():
    if not os.path.isdir("/kaggle/input"):
        print("no /kaggle/input")
        return
    for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
        depth = root[len("/kaggle/input"):].count(os.sep)
        if depth <= 4:
            print("  " * depth + (os.path.basename(root) or "/kaggle/input") + f"/  ({len(files)} files)")


print("attached inputs:")
show_inputs()

In [ ]:
# LLM stack in its own environment (transformers >= 5.12 for Gemma-4); Kaggle's torch and
# CUDA are reused via --system-site-packages. The repo's MuRIL/SVM code stays on the
# system environment.
LLM_PY = "/tmp/llmenv/bin/python"
if not os.path.exists(LLM_PY):
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", "--without-pip",
                    "/tmp/llmenv"], check=True)
subprocess.run(f'{LLM_PY} -m pip install -q "transformers>=5.12,<6" "peft>=0.17" '
               f'"bitsandbytes>=0.46" "accelerate>=1.0" emoji ftfy sentencepiece protobuf',
               shell=True, check=True)
subprocess.run([LLM_PY, "-c", "import transformers, peft, bitsandbytes; print('LLM env: "
                "transformers', transformers.__version__, '| peft', peft.__version__)"], check=True)

MODEL = "google/gemma-4-12B"
S = "gemma-4-12b"


def download(model=MODEL):
    from huggingface_hub import snapshot_download
    snapshot_download(model, cache_dir=HF_CACHE, token=os.environ.get("HF_TOKEN") or None,
                      allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt",
                                      "tokenizer*", "*.jinja"])


def llm_job(name, train, predict, seed, evalf="", extra=""):
    ev = f"--eval {evalf}" if evalf else ""
    return {"name": name, "log": str(LOGS / f"{name}.log"),
            "cmd": (f"{LLM_PY} -u -m hastika.task_b.llm_classifier --task {TASK} --model {MODEL} "
                    f"--train {train} {ev} --predict {' '.join(map(str, predict))} "
                    f"--out {RUNS / name} --seed {seed} --cache {HF_CACHE} {extra}")}


def tapt_job(name, texts, extra=""):
    """Gemma TAPT: LoRA next-token training on unlabelled comments (the MuRIL +2.9 lesson)."""
    return {"name": name, "log": str(LOGS / f"{name}.log"),
            "cmd": (f"{LLM_PY} -u -m hastika.task_b.llm_tapt --model {MODEL} "
                    f"--texts {' '.join(map(str, texts))} --out {RUNS / name} --cache {HF_CACHE} {extra}")}


def load_probs(run_dir, csv):
    """<input stem>_probs.npy, or None when it is missing, belongs to other rows (another
    task's output found among the inputs), or is collapsed onto one class (Task A Run 28's
    seed 43 predicted Non-Hate for every row)."""
    run_dir, csv = pathlib.Path(run_dir), pathlib.Path(csv)
    p = run_dir / f"{csv.stem}_probs.npy"
    if not p.exists():
        return None
    ids = pd.read_csv(run_dir / f"{csv.stem}_ids.csv")["id"].tolist()
    if ids != pd.read_csv(csv)["id"].tolist():
        return None
    probs = np.load(p)
    share = np.bincount(probs.argmax(1), minlength=probs.shape[1]) / len(probs)
    if share.max() > 0.9:
        print(f"skipping {run_dir.name}: collapsed, {share.max():.0%} of rows in one class")
        return None
    return probs


def run_jobs(jobs, on_done=None):
    """run_queue with two automatic retries per job: exit code 4 (collapsed after epoch 1)
    reruns with seed + 1000; out of GPU memory reruns at micro-batch 4 (same effective
    batch). on_done sees each job's final outcome."""
    import re
    registry, retried = {j["name"]: j for j in jobs}, set()

    def done(name, code):
        job, nxt = registry.get(name), []
        if code and job is not None and name not in retried:
            log = open(job["log"], errors="replace").read()
            seed = re.search(r"--seed (\d+)", job["cmd"])
            if code == 4 and seed:
                print(f"{name} collapsed -- rerunning with seed {int(seed.group(1)) + 1000}")
                nxt = [{**job, "cmd": job["cmd"] + f" --seed {int(seed.group(1)) + 1000}"}]
            elif "OutOfMemoryError" in log and "llm_tapt" in job["cmd"]:
                # TAPT already runs at micro-batch 1 by default: last resort is shorter text
                print(f"{name} ran out of GPU memory -- rerunning TAPT at 64 tokens")
                nxt = [{**job, "cmd": job["cmd"] + " --bs 1 --grad-accum 16 --max-len 64 --logit-chunk 8"}]
            elif "OutOfMemoryError" in log:
                print(f"{name} ran out of GPU memory -- rerunning at micro-batch 4")
                nxt = [{**job, "cmd": job["cmd"] + " --bs 4 --grad-accum 4 --eval-bs 8"}]
            if nxt:
                retried.add(name)
        if not nxt and on_done is not None:
            nxt = list(on_done(name, code) or [])
        for j in nxt:
            registry[j["name"]] = j
        return nxt

    return run_queue(jobs, N_GPU, raise_on_fail=False, on_done=done, stop_at=HARD_STOP)


def write_zip(name, probs, recommended=False):
    import zipfile
    assert probs.shape == (len(test), len(CLASSES)), probs.shape
    pred = pd.DataFrame({"id": test["id"], "label": [CLASSES[i] for i in probs.argmax(1)]})
    assert pred["id"].is_unique and set(pred["label"]) <= set(CLASSES)
    np.save(SUB / f"{name}_test_probs.npy", probs)
    csv = SUB / f"predictions_{name}.csv"
    pred.to_csv(csv, index=False)
    path = SUB / f"{'RECOMMENDED_' if recommended else ''}{name}.zip"
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(csv, "predictions.csv")
    prior = df[LABEL_COL].value_counts(normalize=True)
    counts = pred["label"].value_counts(normalize=True)
    print(f"{path.name}  predicted/prior: "
          + "  ".join(f"{c[:5]} {counts.get(c, 0):.1%}/{prior[c]:.1%}" for c in CLASSES))
    return path


download()

In [ ]:
QWEN = "Qwen/Qwen3-8B-Base"     # second-best reader of Kanglish in Run 20's screen (3.19 bits/char)
GEMMA = MODEL
SEEDS = [42, 43]
HO = dict(train=SPLIT / "train.csv", predict=[SPLIT / "holdout.csv", TEST_CSV], evalf=SPLIT / "holdout.csv")
TAPT_DIR = RUNS / "tapt_train"

# Everything is trained here; nothing needs attaching.
#   reference: Gemma TAPT x2 -- the final recipe (TAPT 1 epoch on the training split + external
#              corpus, then 3-epoch classifiers from that adapter)
#   candidate: the same two Gemma models + two Qwen3-8B models, equal weight per model
download(QWEN)
MODEL = QWEN                                   # llm_job reads MODEL when each job is built
qwen_jobs = [llm_job(f"qwen_s{s}", seed=s, **HO) for s in SEEDS]
MODEL = GEMMA
gemma_jobs = [llm_job(f"gemma_tapt_s{s}", seed=s, extra=f"--init-adapter {TAPT_DIR}", **HO) for s in SEEDS]
state = {"tapt_ok": False}


def on_done(name, code):
    if name == "tapt_train":
        state["tapt_ok"] = code == 0
        if code:
            print(f"TAPT FAILED (exit {code}) -- the Gemma reference cannot be trained; see {LOGS / 'tapt_train.log'}")
            return []
        return gemma_jobs
    return []


# GPU 0: TAPT, then a Gemma classifier; GPU 1: both Qwen fits, then the other Gemma classifier
jobs = [tapt_job("tapt_train", [SPLIT / "train.csv", EXTERNAL])] + qwen_jobs
codes = run_jobs(jobs, on_done)
if not state["tapt_ok"]:
    raise SystemExit("TAPT did not complete -- no Gemma reference, so no comparison. See the TAPT log above.")
print("exit codes:", codes, f"| {(time.time() - T_START) / 3600:.1f} h")

In [ ]:
qwen = {s: load_probs(RUNS / f"qwen_s{s}", SPLIT / "holdout.csv") for s in SEEDS}
qwen = {s: p for s, p in qwen.items() if p is not None}
ref = {s: load_probs(RUNS / f"gemma_tapt_s{s}", SPLIT / "holdout.csv") for s in SEEDS}
ref = {s: p for s, p in ref.items() if p is not None}
assert qwen and ref, "missing Qwen or Gemma holdout probabilities -- see the logs"
for s, p in qwen.items():
    report(y_ho, p, f"Qwen seed {s}")
for s, p in ref.items():
    report(y_ho, p, f"Gemma TAPT seed {s}")
q = np.mean(list(qwen.values()), 0)
g = np.mean(list(ref.values()), 0)
# pre-registered: equal weight per model, exactly as combine_probs builds the final
both = np.mean(list(ref.values()) + list(qwen.values()), 0)
scores = {"qwen": report(y_ho, q, f"Qwen x{len(qwen)}"),
          "gemma_tapt": report(y_ho, g, f"Gemma TAPT x{len(ref)}"),
          "gemma+qwen": report(y_ho, both, f"Gemma TAPT + Qwen, {len(ref) + len(qwen)} models")}
print("for context, recorded earlier on these rows: Gemma TAPT x2 0.8619-0.8675, Run 36's 4-model mix 0.8721")
dis = float((q.argmax(1) != g.argmax(1)).mean())
print(f"\nQwen and Gemma disagree on {dis:.1%} of holdout rows")
b = boot_line("Gemma+Qwen - Gemma", both, g)
verdict = ("SIGNIFICANT gain -- build the full-data Qwen fit" if b["ci95"][0] > 0 else
           "probable gain, but inside the noise -- optional" if b["p_better"] >= 0.7 else
           "no gain -- stop here, keep Gemma only")
print(f"\nverdict: {verdict} (P(better) {b['p_better']:.2f})")
test_q = [load_probs(RUNS / f"qwen_s{s}", TEST_CSV) for s in SEEDS]
test_q = [p for p in test_q if p is not None]
if test_q:
    write_zip("a40_qwen_holdout_models", np.mean(test_q, 0))
json.dump({"commit": head, "scores": scores, "disagreement": dis, "bootstrap": b,
           "verdict": verdict, "exit_codes": codes}, open(OUT / "result.json", "w"), indent=2)